# OASIS-1 dementia ViT: positive control (DECISION_RULES.md §12.1)
Settings: **Internet ON**, **Accelerator: GPU T4 x2** (one GPU is used). Attach the **same dataset as the feature notebook** (made from `oasis1_vit_input.zip`).
Run with **Save Version → Save & Run All** (about 20–30 min). Download `/kaggle/working/` (or the version's Output tab):
`pc_features/<model>__<scenario>.npz` + `pc_validity.csv`. Put them in `results/positive_control/kaggle/` locally.

A signal of known location is planted in the images of synthetic-positive slices (labels permuted, §12.1):
`plant_mtl` (MTL voxels × 0.8), `plant_global` (all brain voxels × 0.9), `plant_mtl_weak` (MTL voxels × 0.9, added in the change log).
Strengths were fixed by the calibration in `results/positive_control/calibration.csv` before any lesion run.
No probe is fitted here.

In [ ]:
!pip install -q timm==1.0.30
import os, sys, time, glob
import numpy as np, pandas as pd, torch, torch.nn.functional as F, timm

DATA = os.path.dirname(glob.glob("/kaggle/input/**/meta.csv", recursive=True)[0])
sys.path.insert(0, DATA)
from vitprobe.models import ProbedViT, REGISTRY
from vitprobe.masks import token_labels, isolate_mask, edge_keep_mask, n_cross_pairs

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("data:", DATA, "| timm", timm.__version__, "| torch", torch.__version__, "| device", DEV)
OUT = "/kaggle/working/pc_features"
os.makedirs(OUT, exist_ok=True)

images = torch.from_numpy(np.load(f"{DATA}/images.npy").astype(np.float32))   # (990, 145, 145) in [0, 1]
masks = {r: torch.from_numpy(np.load(f"{DATA}/{r}.npy").astype(np.float32)) for r in ("mtl", "ventricles", "brain")}
N = images.shape[0]

PLANT_HEX = "003fff83e007fe0003e007fe0f800000000f83e007fe0003e0f83e0003ffffc1f07c1f003ff07fe0f83ff07fff0001f07c0007c1f003e0f8000f801f0000007c00f83fffffe0ffffff83ff003e007c00fffff07c0007c1ffffe007c0007fff07fe007fe007fff07fe0f83fffffffffffff8000003e0f8000ffc1fffc"
plant = np.unpackbits(np.frombuffer(bytes.fromhex(PLANT_HEX), np.uint8))[:N].astype(bool)
assert plant.sum() == 500, plant.sum()
POS, NEG = np.where(plant)[0], np.where(~plant)[0]

MODELS = ["dinov2_s", "dinov2_s_reg", "dinov2_b", "dinov2_b_reg", "deit_s"]
SCENARIOS = {"plant_mtl": ("mtl", 0.20), "plant_global": ("brain", 0.10), "plant_mtl_weak": ("mtl", 0.10)}
REGIONS = ["mtl", "ventricles"]          # same order as the main run, so the random-cut seeds match (§7)
TH, BRAIN_TH, SEEDS, BATCH = 0.25, 0.25, (0, 1, 2), 64
print("slices", N, "| synthetic positive", len(POS), "| negative", len(NEG))

In [ ]:
# --- same as kaggle/make_vit_nb.py (seeding included), restricted to the §12.1 conditions ---
def to_input(x, net):
    x = F.interpolate(x[:, None], size=(224, 224), mode="bilinear", align_corners=False).clamp(0, 1)
    x = x.repeat(1, 3, 1, 1)
    mean = torch.tensor(net.mean).view(1, 3, 1, 1)
    std = torch.tensor(net.std).view(1, 3, 1, 1)
    return ((x - mean) / std).to(DEV)


def patch_frac(m, idx, patch):
    return F.avg_pool2d(F.interpolate(m[idx][:, None], size=(224, 224), mode="nearest"), patch).flatten(1)


def gens(seed, idx):
    return [torch.Generator().manual_seed(seed * 1_000_003 + int(i)) for i in idx]


def tok(patch_side, n_prefix):
    return token_labels(patch_side.to(DEV), n_prefix, "fg")


def random_patches(region, brain, idx, seed):
    region, brain = region.cpu(), brain.cpu()
    sel = torch.zeros_like(region)
    for b, g in enumerate(gens(seed, idx)):
        cand = torch.nonzero(brain[b] & ~region[b]).flatten()
        k = min(int(region[b].sum()), len(cand))
        if k:
            sel[b, cand[torch.randperm(len(cand), generator=g)[:k]]] = True
    return sel


def conditions(idx, fr, brain, n_prefix):
    yield "full", None
    for ri, r in enumerate(REGIONS):
        R = fr[r] >= TH
        lab = tok(~R, n_prefix)
        yield f"remove_{r}", isolate_mask(lab)
        for s in SEEDS:
            base = 100 * ri + s
            yield (f"random_links_{r}#{s}",
                   edge_keep_mask(lab, None, "random", generators=gens(base, idx), n_remove=n_cross_pairs(lab)))
            yield f"random_patches_{r}#{s}", isolate_mask(tok(~random_patches(R, brain, idx, base + 10), n_prefix))


def extract(net, idx_all, plant_spec=None):
    feats, counts = {}, {"n_mtl": [], "n_ventricles": [], "n_brain": []}
    for s in range(0, len(idx_all), BATCH):
        idx = torch.as_tensor(idx_all[s:s + BATCH])
        x = images[idx]
        if plant_spec is not None:
            region, alpha = plant_spec
            x = x * (1 - alpha * masks[region][idx])
        x = to_input(x, net)
        fr = {r: patch_frac(masks[r], idx, net.patch) for r in REGIONS}
        brain = patch_frac(masks["brain"], idx, net.patch) >= BRAIN_TH
        for r in REGIONS:
            counts[f"n_{r}"] += (fr[r] >= TH).sum(1).tolist()
        counts["n_brain"] += brain.sum(1).tolist()
        for name, keep in conditions(idx, fr, brain, net.n_prefix):
            net.ctrl.keep, net.ctrl.layers = keep, None
            with torch.no_grad():
                feats.setdefault(name, []).append(net.features(x).float().cpu().numpy().astype(np.float16))
        net.ctrl.keep = None
    return {k: np.concatenate(v) for k, v in feats.items()}, {k: np.array(v) for k, v in counts.items()}


def merge(neg, pos):
    out = {}
    for k in neg:
        a = np.zeros((N,) + neg[k].shape[1:], neg[k].dtype)
        a[NEG], a[POS] = neg[k], pos[k]
        out[k] = a
    return out

In [ ]:
validity = []
for key in MODELS:
    t0 = time.time()
    net = ProbedViT(key).to(DEV)
    timm_id, _, kw = REGISTRY[key]
    stock = timm.create_model(timm_id, pretrained=True, **kw).eval().to(DEV)
    with torch.no_grad():
        x = to_input(images[:32], net)
        diff = (net.features(x) - stock.forward_head(stock.forward_features(x), pre_logits=True)).abs().max().item()
    del stock
    neg, neg_cnt = extract(net, NEG)                       # unchanged images, shared by all scenarios
    for sc, spec in SCENARIOS.items():
        path = f"{OUT}/{key}__{sc}.npz"
        if os.path.exists(path):
            print(key, sc, "already done"); continue
        pos, pos_cnt = extract(net, POS, spec)
        np.savez_compressed(path, **merge(neg, pos), **merge(neg_cnt, pos_cnt), alpha=np.array(spec[1]))
        print(key, sc, "saved", flush=True)
    validity.append(dict(model=key, patched_vs_stock_maxabs=diff, seconds=round(time.time() - t0)))
    print(validity[-1], flush=True)
    del net
    torch.cuda.empty_cache()

pd.DataFrame(validity).to_csv("/kaggle/working/pc_validity.csv", index=False)
print(pd.DataFrame(validity))
print(sorted(os.listdir(OUT)))